# Aptia — POC de Fast Prompting
### Entrega 2: Fast Prompting en Acción — Desentrañando la Magia

**Autor:** Orlando Bermúdez · **Curso:** Inteligencia Artificial: Generación de Prompts · **Comisión:** #96165

Esta notebook implementa una prueba de concepto (POC) de **Aptia**, la plataforma que ayuda a
propietarios de motocicletas en Valledupar a encontrar repuestos homologables y recomendaciones
de mantenimiento, usando técnicas de *fast prompting* sobre la API de Anthropic (Claude).

**Técnicas de fast prompting aplicadas en esta notebook:**

- Role prompting (prompt de sistema por función)
- Salida estructurada (JSON con esquema fijo)
- Few-shot prompting (chatbot)
- Parametrización de prompts (marcas y categorías variables)
- Guardrails explícitos contra alucinaciones
- Control de temperatura según el tipo de tarea
- Caché en memoria para minimizar consultas repetidas a la API

Cada función realiza **una única llamada a la API por consulta**, ver la sección de
[Análisis de costos](#Análisis-de-costos) al final de esta notebook.


## 1. Instalación e importación de librerías

In [ ]:
# Si es la primera vez que ejecutas esta notebook, instala las dependencias:
# !pip install anthropic pandas python-dotenv

import os
import json
import getpass

import pandas as pd
import anthropic


## 2. Configuración segura de la API key

La API key nunca se escribe directamente en el código. Se busca primero en una variable de entorno (`ANTHROPIC_API_KEY`) y, si no existe, se solicita de forma oculta con `getpass`.

In [ ]:
api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    api_key = getpass.getpass("Ingresa tu Anthropic API key: ")

client = anthropic.Anthropic(api_key=api_key)

MODELO = "claude-sonnet-5"  # Puede cambiarse por claude-haiku-4-5-20251001 para reducir aún más el costo


## 3. Función central de consulta (con caché)

Todas las funciones de Aptia pasan por esta única función para hablar con la API.
Esto permite:

1. Contar cuántas llamadas reales se hacen (para el análisis de costos).
2. Cachear resultados: si se repite exactamente la misma consulta, no se vuelve a llamar a la API.
3. Centralizar el manejo de errores.


In [ ]:
contador_llamadas_api = 0
cache_respuestas = {}


def consultar_claude(system_prompt: str, user_prompt: str, max_tokens: int = 600, temperature: float = 0.3) -> str:
    """
    Realiza UNA única consulta a la API de Claude, con caché en memoria.

    Args:
        system_prompt: define el rol del modelo (role prompting).
        user_prompt: la consulta específica del usuario, ya con sus parámetros insertados.
        max_tokens: límite de tokens de salida (se ajusta por función para no pagar de más).
        temperature: control de creatividad vs. consistencia.

    Returns:
        El texto de la respuesta del modelo.
    """
    global contador_llamadas_api

    clave_cache = (system_prompt, user_prompt, max_tokens, temperature)
    if clave_cache in cache_respuestas:
        print("Respuesta obtenida de la caché (0 llamadas nuevas a la API).")
        return cache_respuestas[clave_cache]

    respuesta = client.messages.create(
        model=MODELO,
        max_tokens=max_tokens,
        temperature=temperature,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )

    contador_llamadas_api += 1
    texto = respuesta.content[0].text
    cache_respuestas[clave_cache] = texto
    return texto


def extraer_json(texto: str):
    """
    Extrae y parsea JSON de la respuesta del modelo, incluso si viene
    envuelto en bloques de código Markdown (```json ... ```).
    """
    texto_limpio = texto.strip().replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(texto_limpio)
    except json.JSONDecodeError:
        print("No se pudo parsear la respuesta como JSON. Respuesta cruda:")
        print(texto)
        return None


## 4. Función 1 — Ficha de homologación de repuestos

Basada en el Prompt 1 de la Preentrega 1, ahora rediseñado con:

- **Salida estructurada (JSON)** en vez de tabla en texto libre, para poder graficarla con `pandas`.
- **Parametrización** de marcas y categorías (listas, no texto fijo).
- **Guardrail** explícito: el modelo debe declarar `"confianza"` y no inventar datos.


In [ ]:
def ficha_homologacion(modelo_moto: str, marcas: list, categorias: list) -> pd.DataFrame:
    """
    Genera, con UNA sola consulta a la API, una ficha de homologación de repuestos
    para el modelo de moto indicado, y la devuelve como DataFrame de pandas.
    """
    system_prompt = (
        "Eres un mecánico experto en motocicletas de baja y media cilindrada, con conocimiento "
        "profundo de las marcas vendidas en Colombia. Respondes siempre con datos verificables; "
        "si no tienes certeza sobre una homologación, lo declaras explícitamente en el campo "
        "'confianza' en vez de inventar información, porque un dato erróneo puede afectar la "
        "seguridad del usuario."
    )

    user_prompt = f"""
Genera una ficha de homologación de repuestos para el modelo de moto: {modelo_moto}.

Marcas de referencia para buscar compatibilidad: {", ".join(marcas)}.
Categorías de repuesto a evaluar: {", ".join(categorias)}.

Responde EXCLUSIVAMENTE con un JSON válido (sin texto adicional, sin explicaciones fuera del
JSON), con esta estructura exacta:

{{
  "modelo": "{modelo_moto}",
  "repuestos": [
    {{
      "categoria": "nombre de la categoría",
      "compatible_con": "modelos/marcas compatibles",
      "diferencias": "diferencias a considerar (calidad, durabilidad, ajuste)",
      "confianza": "alta | media | baja"
    }}
  ]
}}
""".strip()

    texto_respuesta = consultar_claude(system_prompt, user_prompt, max_tokens=800, temperature=0.2)
    datos = extraer_json(texto_respuesta)

    if datos is None:
        return pd.DataFrame()

    return pd.DataFrame(datos["repuestos"])


# --- Demostración ---
marcas_colombia = ["AKT", "Yamaha", "Honda", "Suzuki", "Bajaj", "TVS", "Hero"]
categorias_base = ["pastillas de freno", "bujía", "filtro de aceite", "kit de arrastre", "llantas"]

df_homologacion = ficha_homologacion("AKT NKD 125", marcas_colombia, categorias_base)
df_homologacion


## 5. Función 2 — Recomendación de aceite y llantas

También con salida estructurada, temperatura baja (0.2) para maximizar consistencia.

In [ ]:
def recomendacion_mantenimiento(modelo_moto: str, cilindraje_cc: int, uso: str, km_mensuales: int) -> dict:
    """
    Genera, con UNA sola consulta a la API, una recomendación de aceite y llantas
    personalizada según el uso real de la moto.
    """
    system_prompt = (
        "Eres un asesor técnico de mantenimiento de motocicletas. Tus recomendaciones son "
        "prácticas, breves y siempre recuerdan que ante cualquier duda se debe consultar a un "
        "taller de confianza."
    )

    user_prompt = f"""
Datos del usuario:
- Modelo de moto: {modelo_moto}
- Cilindraje: {cilindraje_cc}cc
- Uso principal: {uso}
- Kilometraje mensual aproximado: {km_mensuales} km

Responde EXCLUSIVAMENTE con un JSON válido con esta estructura exacta:

{{
  "aceite_recomendado": "tipo y viscosidad, y frecuencia de cambio",
  "llanta_recomendada": "tipo de llanta según el uso",
  "alerta_mantenimiento": "alerta breve si el uso exige mantenimiento más frecuente, o null si no aplica"
}}
""".strip()

    texto_respuesta = consultar_claude(system_prompt, user_prompt, max_tokens=400, temperature=0.2)
    return extraer_json(texto_respuesta)


# --- Demostración ---
recomendacion = recomendacion_mantenimiento(
    modelo_moto="AKT NKD 125",
    cilindraje_cc=125,
    uso="mototaxismo",
    km_mensuales=2500,
)
recomendacion


## 6. Función 3 — Chatbot de consultas frecuentes (few-shot)

Se incluyen ejemplos de pregunta-respuesta directamente en el *system prompt* (few-shot
prompting) para fijar el tono, el largo y el disclaimer esperado, sin necesidad de que el
modelo los infiera desde cero en cada consulta.


In [ ]:
EJEMPLOS_FEW_SHOT = """
Ejemplo 1:
Usuario: ¿Cada cuánto debo cambiar el aceite si uso la moto para trabajar todos los días?
Aptia: ¡De una! Si le das uso diario e intenso, lo ideal es cambiarlo cada 1.000-1.500 km, no
esperes a los 3.000 km del manual. Recuerda que esto es orientativo, así que llévala a un taller
de confianza para confirmar según tu modelo.

Ejemplo 2:
Usuario: ¿Le puedo poner llantas de una Pulsar a mi AKT NKD?
Aptia: Depende de la medida del rin, parcero. Si coinciden las medidas sí calzan físicamente,
pero el comportamiento puede variar. Esto es orientativo: confírmalo con un mecánico certificado
antes de comprar.
""".strip()


def chatbot_aptia(pregunta_usuario: str) -> str:
    """
    Responde una pregunta frecuente de un usuario, con UNA sola consulta a la API,
    usando few-shot prompting para fijar tono y formato.
    """
    system_prompt = (
        "Eres el asistente virtual de Aptia, una app que ayuda a los usuarios de motocicleta en "
        "Valledupar a encontrar repuestos homologables y resolver dudas básicas de mantenimiento. "
        "Respondes en máximo 4 líneas, en tono amigable y cercano (puedes usar expresiones típicas "
        "del Caribe colombiano sin exagerar), y SIEMPRE aclaras que la recomendación es orientativa "
        "y no reemplaza la revisión de un mecánico certificado.\n\n"
        f"Estos son ejemplos del tono y formato esperado:\n{EJEMPLOS_FEW_SHOT}"
    )

    return consultar_claude(system_prompt, pregunta_usuario, max_tokens=200, temperature=0.6)


# --- Demostración ---
respuesta_chat = chatbot_aptia("¿Qué tipo de bujía le sirve a una Yamaha FZ 150?")
print(respuesta_chat)


## 7. Celda interactiva

Para cumplir con la recomendación de que los prompts no estén cargados directamente y fijos en
la celda, esta sección permite ingresar los parámetros por teclado (`input()`) en tiempo de
ejecución.


In [ ]:
def demo_interactiva():
    """
    Punto de entrada interactivo: permite elegir qué función de Aptia probar
    e ingresar los parámetros sin modificar el código.
    """
    print("¿Qué querés consultar?")
    print("1. Ficha de homologación de repuestos")
    print("2. Recomendación de aceite y llantas")
    print("3. Chatbot de consultas frecuentes")
    opcion = input("Elegí una opción (1/2/3): ").strip()

    if opcion == "1":
        modelo_moto = input("Modelo de la moto (ej. AKT NKD 125): ").strip()
        resultado = ficha_homologacion(modelo_moto, marcas_colombia, categorias_base)
        display(resultado)

    elif opcion == "2":
        modelo_moto = input("Modelo de la moto: ").strip()
        cilindraje = int(input("Cilindraje en cc: ").strip())
        uso = input("Uso principal (urbano diario / mototaxismo / carretera / mixto): ").strip()
        km = int(input("Kilometraje mensual aproximado: ").strip())
        resultado = recomendacion_mantenimiento(modelo_moto, cilindraje, uso, km)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))

    elif opcion == "3":
        pregunta = input("Escribí tu pregunta: ").strip()
        print(chatbot_aptia(pregunta))

    else:
        print("Opción no válida.")


# Para probarlo, descomenta la siguiente línea y ejecutá la celda:
# demo_interactiva()


## Análisis de costos

In [ ]:
print(f"Total de llamadas reales a la API en esta sesión: {contador_llamadas_api}")
print(f"Consultas cacheadas (sin costo adicional) disponibles: {len(cache_respuestas)}")


Con las tres demostraciones de arriba, la notebook realizó exactamente **3 llamadas a la API**
(una por función), sin llamadas encadenadas ni de verificación. Si se vuelve a ejecutar una
consulta idéntica (mismos parámetros), la respuesta se sirve desde la caché en memoria y el
contador no aumenta, lo que evita pagar dos veces por la misma pregunta.

## 8. Conclusión — Mejoras respecto a la Preentrega 1

Aplicar técnicas de *fast prompting* mejoró la propuesta original en varios sentidos concretos:

- **De texto libre a JSON estructurado:** en la Preentrega 1 los prompts pedían una tabla en
  texto plano. Ahora la salida es JSON parseable, lo que permite integrar la respuesta
  directamente en una futura app (tabla, tarjeta, notificación) sin procesamiento manual.
- **Menos alucinación, más trazabilidad:** el campo `"confianza"` obliga al modelo a
  autoevaluarse en cada respuesta, en vez de simplemente advertir "si no estás seguro, dilo" de
  forma genérica como en la Preentrega 1.
- **Costo controlado desde el diseño:** cada función = 1 llamada a la API, con caché para evitar
  duplicados. Esto responde directamente al riesgo señalado en la consigna: una app que funciona
  pero consulta de más no es rentable.
- **Prompts reutilizables y parametrizados:** las listas de marcas y categorías siguen siendo
  variables (heredado de la Preentrega 1), por lo que escalar Aptia a más marcas colombianas o
  a nuevas categorías de repuesto (válvulas, cadenilla, kit de empaques, balineras) no requiere
  tocar la lógica de las funciones, solo ampliar esas listas.

En conjunto, estas mejoras acercan a Aptia de una prueba de concepto conceptual a una
implementación con una lógica de costos y de confiabilidad pensada para producción.
